In [4]:
AGENTS = [
    "20250603_Refact_Agent_claude-4-sonnet",
    "20250720_Lingxi-v1.5_claude-4-sonnet-20250514",
    "20250805_openhands-Qwen3-Coder-480B-A35B-Instruct",
    "20250928_trae_doubao_seed_code",
    "20250807_mini-v1.7.0_gpt-5-mini",
]

In [5]:
import json
import os
import pandas as pd
import numpy as np
import pprint as pp
from scipy.stats import wilcoxon
from dataset.extract_ground_truths.effect.process_agent_patch import get_diff_info_per_instance
from execution.util import get_instance_ids

In [6]:
resolved_dict = {}
base_path = "/home/yusuf/explainbench/shared_logs/logs/run_evaluation/output_per_step/efficacy"
for agent in AGENTS:
    path = agent + ".json" if "mini-" not in agent else agent + "_resolved.json"
    path = os.path.join(base_path, path)
    with open(path, "r") as f:
        temp = json.load(f)
    resolved_dict[agent] = temp["resolved"]

# INTENT

In [7]:
def extract_score_local_intent(input_dict, agent, q_type, resolved_dict):
    scores_dict = input_dict[agent]
    instance_ids = []
    agents = []
    scores = []
    q_types = []
    trials = []
    is_resolved = []
    answers = []
    for id_, result_dict in scores_dict.items():
        temp_scores = result_dict["individual_scores"]
        for idx, s in enumerate(temp_scores):
            scores.append(s)
            trials.append(idx+1)
            instance_ids.append(id_)
            is_resolved.append(id_ in resolved_dict[agent])
            agents.append(agent)
            q_types.append(q_type)
            answers.append(result_dict["all_pred"][idx][0])
    return agents, instance_ids, scores, q_types, is_resolved, trials, answers

In [8]:
local_intent = "results_intent_local/eval.individual.intent.json"
with open(local_intent, "r") as f:
    local_intent = json.load(f)

In [9]:
instance_ids = []
scores = []
agents = []
q_types = []
trials = []
resolved = []
answers = []
for agent in (AGENTS):
    temp_agents, temp_instance_ids, temp_scores, temp_q_types, temp_resolved, temp_trials, temp_answers = extract_score_local_intent(local_intent, agent, "local_intent", resolved_dict)
    agents.extend(temp_agents)
    q_types.extend(temp_q_types)
    instance_ids.extend(temp_instance_ids)
    scores.extend(temp_scores)
    resolved.extend(temp_resolved)
    trials.extend(temp_trials)
    answers.extend(temp_answers)
df_dict = {
    "agent_name": agents,
    "q_type": q_types,
    "trial": trials,
    "id_": instance_ids,
    "answer": answers,
    "score": scores,
    "resolved": resolved
}        

local_intent = pd.DataFrame(df_dict)

In [10]:
ANSWER_JSON = "/home/yusuf/explainbench/shared_logs/logs/run_evaluation/output_per_step/experiment_w_reachability/step4.intent.json"
with open(ANSWER_JSON, "r") as f:
    answer_json = json.load(f)

answers = []
for idx, row in local_intent.iterrows():
    agent = row["agent_name"]
    id_ = row["id_"]
    answers.append(
        answer_json[agent][id_]["answer"][0])
    
local_intent["truth"] = answers

In [73]:
ee_intent = "results_intent_ee/final_results_intent_pbtassertionmcq.json"
with open(ee_intent, "r") as f:
    ee_intent = json.load(f)

In [74]:
def extract_score_local_ee(input_dict, agent, q_type, resolved_dict):
    scores_dict = input_dict[agent]
    instance_ids = []
    agents = []
    scores = []
    q_types = []
    trials = []
    is_resolved = []
    answers = []
    gts = []
    choices = []
    for id_, result_dict in scores_dict.items():
        temp_scores = result_dict["individual_scores"]
        for idx, s in enumerate(temp_scores):
            scores.append(s)
            trials.append(idx+1)
            instance_ids.append(id_)
            is_resolved.append(id_ in resolved_dict[agent])
            agents.append(agent)
            q_types.append(q_type)
            answers.append(result_dict["all_pred"][idx]["selection"])
            gts.append(result_dict["answer_gt"])
            choices.append(result_dict["choices"])
    return agents, instance_ids, scores, q_types, is_resolved, trials, answers, gts, choices 

In [75]:
instance_ids = []
scores = []
agents = []
q_types = []
trials = []
resolved = []
answers = []
gts = []
choices = []
for agent in (AGENTS):
    temp_agents, temp_instance_ids, temp_scores, temp_q_types, temp_resolved, temp_trials, temp_answers, temp_gts, temp_choices = extract_score_local_ee(ee_intent, agent, "ee_intent", resolved_dict)
    agents.extend(temp_agents)
    q_types.extend(temp_q_types)
    instance_ids.extend(temp_instance_ids)
    scores.extend(temp_scores)
    resolved.extend(temp_resolved)
    trials.extend(temp_trials)
    answers.extend(temp_answers)
    gts.extend(temp_gts)
    choices.extend(temp_choices)
df_dict = {
    "agent_name": agents,
    "q_type": q_types,
    "trial": trials,
    "id_": instance_ids,
    "answer": answers,
    "choices": choices,
    "score": scores,
    "resolved": resolved,
    "truth": gts
}        

ee_intent = pd.DataFrame(df_dict)

In [76]:
answer_to_idx = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}
def is_uninformative(answer, choices):
    idx = answer_to_idx[answer]
    selected = choices[idx]
    if selected == "The question cannot be answered based on the explanation alone.":
        return False
    return True

In [77]:
ee_intent["informative"] = ee_intent.apply(lambda x: is_uninformative(x.answer, x.choices), axis=1)
ee_intent.informative.value_counts()

informative
True     5126
False    2299
Name: count, dtype: int64

In [78]:
ee_intent.head(3)

,agent_name,q_type,trial,id_,answer,choices,score,resolved,truth,informative
0,20250603_Refact_Agent_claude-4-sonnet,ee_intent,1,astropy__astropy-13977,A,[The question cannot be answered based on the ...,False,False,B,False
1,20250603_Refact_Agent_claude-4-sonnet,ee_intent,2,astropy__astropy-13977,A,[The question cannot be answered based on the ...,False,False,B,False
2,20250603_Refact_Agent_claude-4-sonnet,ee_intent,3,astropy__astropy-13977,A,[The question cannot be answered based on the ...,False,False,B,False


In [79]:
ee_intent.drop(columns=["choices"], inplace=True)

In [80]:
local_intent["answer"] =local_intent.answer.str.upper()
local_intent["truth"] =local_intent.truth.str.upper()

In [87]:
local_intent["informative"] = local_intent.apply(lambda x: x.answer != "E", axis=1)

In [88]:
local_intent.truth.value_counts()

truth
A    1925
D    1900
C    1900
B    1700
Name: count, dtype: int64

In [89]:
local_intent.answer.value_counts()

answer
E    2267
D    1471
C    1371
A    1197
B    1119
Name: count, dtype: int64

In [90]:
local_intent.informative.value_counts()

informative
True     5158
False    2267
Name: count, dtype: int64

In [91]:
local_intent.head(3)

,agent_name,q_type,trial,id_,answer,score,resolved,truth,informative
0,20250603_Refact_Agent_claude-4-sonnet,local_intent,1,django__django-11179,D,1.0,True,D,True
1,20250603_Refact_Agent_claude-4-sonnet,local_intent,2,django__django-11179,D,1.0,True,D,True
2,20250603_Refact_Agent_claude-4-sonnet,local_intent,3,django__django-11179,D,1.0,True,D,True


In [92]:
intent = pd.concat([local_intent, ee_intent])
intent

,agent_name,q_type,trial,id_,answer,score,resolved,truth,informative
0,20250603_Refact_Agent_claude-4-sonnet,local_intent,1,django__django-11179,D,1.0,True,D,True
1,20250603_Refact_Agent_claude-4-sonnet,local_intent,2,django__django-11179,D,1.0,True,D,True
2,20250603_Refact_Agent_claude-4-sonnet,local_intent,3,django__django-11179,D,1.0,True,D,True
3,20250603_Refact_Agent_claude-4-sonnet,local_intent,4,django__django-11179,D,1.0,True,D,True
4,20250603_Refact_Agent_claude-4-sonnet,local_intent,5,django__django-11179,D,1.0,True,D,True
...,...,...,...,...,...,...,...,...,...
7420,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,1,pydata__xarray-6938,A,0.0,False,D,False
7421,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,2,pydata__xarray-6938,A,0.0,False,D,False
7422,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,3,pydata__xarray-6938,A,0.0,False,D,False
7423,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,4,pydata__xarray-6938,B,0.0,False,D,True


In [93]:
intent["label"] = "not-informative"
intent["label"] = intent.apply(lambda x: "misaligned" if x.informative and np.allclose(x.score, 0.0) else x.label, axis=1)
intent["label"] = intent.apply(lambda x: "aligned" if x.informative and np.allclose(x.score, 1.0) else x.label, axis=1)

In [94]:
intent.label.value_counts()

label
aligned            7371
not-informative    4566
misaligned         2913
Name: count, dtype: int64

In [95]:
intent.informative.value_counts()

informative
True     10284
False     4566
Name: count, dtype: int64

In [102]:
label_mean = (
    intent.groupby(["agent_name", "q_type"])["label"]
          .value_counts(normalize=True)
          .rename("mean")
          .reset_index()
)

label_mean = label_mean[label_mean.label != "aligned"]

In [106]:
label_mean.sort_values(by=["agent_name", "q_type", "label"]).round(3)

,agent_name,q_type,label,mean
2,20250603_Refact_Agent_claude-4-sonnet,ee_intent,misaligned,0.040
1,20250603_Refact_Agent_claude-4-sonnet,ee_intent,not-informative,0.237
3,20250603_Refact_Agent_claude-4-sonnet,local_intent,misaligned,0.374
5,20250603_Refact_Agent_claude-4-sonnet,local_intent,not-informative,0.271
8,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,ee_intent,misaligned,0.046
7,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,ee_intent,not-informative,0.250
10,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,local_intent,misaligned,0.362
11,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,local_intent,not-informative,0.269
14,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,ee_intent,misaligned,0.032
13,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,ee_intent,not-informative,0.245


In [23]:
# step 1: mean across trials for each (agent, id_)
per_id = (
    intent.groupby(["agent_name", "id_"], as_index=False)["score"]
      .mean()
      .rename(columns={"score": "score_per_id"})
)

# step 2: mean across ids for each agent
agent_score_intent = (
    per_id.groupby("agent_name", as_index=False)["score_per_id"]
          .mean()
          .rename(columns={"score_per_id": "intent_score"})
)

agent_score_intent

,agent_name,intent_score
0,20250603_Refact_Agent_claude-4-sonnet,0.538721
1,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,0.536364
2,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,0.537710
3,20250807_mini-v1.7.0_gpt-5-mini,0.384848
4,20250928_trae_doubao_seed_code,0.484175


# EFFECT

In [26]:
local_effect = "results_effect_local/eval.individual.effect.json"

In [27]:
with open(local_effect, "r") as f:
    local_effect = json.load(f)

In [28]:
instance_ids = []
scores = []
agents = []
q_types = []
trials = []
resolved = []
answers = []
for agent in (AGENTS):
    temp_agents, temp_instance_ids, temp_scores, temp_q_types, temp_resolved, temp_trials, temp_answers = extract_score_local_intent(local_effect, agent, "local_effect", resolved_dict)
    agents.extend(temp_agents)
    q_types.extend(temp_q_types)
    instance_ids.extend(temp_instance_ids)
    scores.extend(temp_scores)
    resolved.extend(temp_resolved)
    trials.extend(temp_trials)
    answers.extend(temp_answers)
df_dict = {
    "agent_name": agents,
    "q_type": q_types,
    "trial": trials,
    "id_": instance_ids,
    "answer": answers,
    "score": scores,
    "resolved": resolved
}        

local_effect = pd.DataFrame(df_dict)

In [29]:
ANSWER_JSON = "/home/yusuf/explainbench/shared_logs/logs/run_evaluation/output_per_step/experiment_w_reachability/step4.json"
with open(ANSWER_JSON, "r") as f:
    answer_json = json.load(f)


answers = []
for idx, row in local_effect.iterrows():
    agent = row["agent_name"]
    id_ = row["id_"]
    answers.append(
        answer_json[agent][id_]["answer"][0])
    
local_effect["truth"] = answers

In [30]:
local_effect["answer"] = local_effect.answer.str.upper()
local_effect["truth"] = local_effect.truth.str.upper()

In [47]:
ee_effect = "results_effect_ee/final_results_effect_pbtresultmcq.json"
with open(ee_effect, "r") as f:
    ee_effect = json.load(f)

In [48]:
def extract_score_local_ee(input_dict, agent, q_type, resolved_dict):
    scores_dict = input_dict[agent]
    instance_ids = []
    agents = []
    scores = []
    q_types = []
    trials = []
    is_resolved = []
    answers = []
    gts = []
    for id_, result_dict in scores_dict.items():
        temp_scores = result_dict["individual_scores"]
        for idx, s in enumerate(temp_scores):
            scores.append(s)
            trials.append(idx+1)
            instance_ids.append(id_)
            is_resolved.append(id_ in resolved_dict[agent])
            agents.append(agent)
            q_types.append(q_type)
            answers.append([result_dict["all_pred"][idx]["before_selection"], result_dict["all_pred"][idx]["after_selection"]])
            gts.append(result_dict["answer_gt"])
    return agents, instance_ids, scores, q_types, is_resolved, trials, answers, gts 

In [49]:
instance_ids = []
scores = []
agents = []
q_types = []
trials = []
resolved = []
answers = []
gts = []
for agent in (AGENTS):
    temp_agents, temp_instance_ids, temp_scores, temp_q_types, temp_resolved, temp_trials, temp_answers, temp_gts = extract_score_local_ee(ee_effect, agent, "ee_effect", resolved_dict)
    agents.extend(temp_agents)
    q_types.extend(temp_q_types)
    instance_ids.extend(temp_instance_ids)
    scores.extend(temp_scores)
    resolved.extend(temp_resolved)
    trials.extend(temp_trials)
    answers.extend(temp_answers)
    gts.extend(temp_gts)
df_dict = {
    "agent_name": agents,
    "q_type": q_types,
    "trial": trials,
    "id_": instance_ids,
    "answer": answers,
    "score": scores,
    "resolved": resolved,
    "truth": gts
}        

ee_effect = pd.DataFrame(df_dict)

In [50]:
ee_effect

,agent_name,q_type,trial,id_,answer,score,resolved,truth
0,20250603_Refact_Agent_claude-4-sonnet,ee_effect,1,django__django-12304,"[A, A]",False,True,"[D, E]"
1,20250603_Refact_Agent_claude-4-sonnet,ee_effect,2,django__django-12304,"[A, A]",False,True,"[D, E]"
2,20250603_Refact_Agent_claude-4-sonnet,ee_effect,3,django__django-12304,"[A, A]",False,True,"[D, E]"
3,20250603_Refact_Agent_claude-4-sonnet,ee_effect,4,django__django-12304,"[A, A]",False,True,"[D, E]"
4,20250603_Refact_Agent_claude-4-sonnet,ee_effect,5,django__django-12304,"[A, A]",False,True,"[D, E]"
...,...,...,...,...,...,...,...,...
7420,20250807_mini-v1.7.0_gpt-5-mini,ee_effect,1,sympy__sympy-20916,"[C, E]",False,False,"[C, C]"
7421,20250807_mini-v1.7.0_gpt-5-mini,ee_effect,2,sympy__sympy-20916,"[C, E]",False,False,"[C, C]"
7422,20250807_mini-v1.7.0_gpt-5-mini,ee_effect,3,sympy__sympy-20916,"[C, E]",False,False,"[C, C]"
7423,20250807_mini-v1.7.0_gpt-5-mini,ee_effect,4,sympy__sympy-20916,"[C, E]",False,False,"[C, C]"


In [51]:
effect = pd.concat([local_effect, ee_effect])

In [53]:
# step 1: mean across trials for each (agent, id_)
per_id = (
    effect.groupby(["agent_name", "id_"], as_index=False)["score"]
      .mean()
      .rename(columns={"score": "score_per_id"})
)

# step 2: mean across ids for each agent
agent_score_effect = (
    per_id.groupby("agent_name", as_index=False)["score_per_id"]
          .mean()
          .rename(columns={"score_per_id": "effect_score"})
)

agent_score_effect.round(3)

,agent_name,effect_score
0,20250603_Refact_Agent_claude-4-sonnet,0.618
1,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,0.618
2,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,0.621
3,20250807_mini-v1.7.0_gpt-5-mini,0.475
4,20250928_trae_doubao_seed_code,0.614


In [54]:
df = pd.concat([effect, intent])

In [55]:
df

,agent_name,q_type,trial,id_,answer,score,resolved,truth
0,20250603_Refact_Agent_claude-4-sonnet,local_effect,1,astropy__astropy-13579,E,0.0,True,C
1,20250603_Refact_Agent_claude-4-sonnet,local_effect,2,astropy__astropy-13579,E,0.0,True,C
2,20250603_Refact_Agent_claude-4-sonnet,local_effect,3,astropy__astropy-13579,E,0.0,True,C
3,20250603_Refact_Agent_claude-4-sonnet,local_effect,4,astropy__astropy-13579,E,0.0,True,C
4,20250603_Refact_Agent_claude-4-sonnet,local_effect,5,astropy__astropy-13579,E,0.0,True,C
...,...,...,...,...,...,...,...,...
7420,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,1,pydata__xarray-6938,A,0.0,False,D
7421,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,2,pydata__xarray-6938,A,0.0,False,D
7422,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,3,pydata__xarray-6938,A,0.0,False,D
7423,20250807_mini-v1.7.0_gpt-5-mini,ee_intent,4,pydata__xarray-6938,B,0.0,False,D


In [56]:
# step 1: mean across trials for each (agent, id_)
per_id = (
    df.groupby(["agent_name", "id_"], as_index=False)["score"]
      .mean()
      .rename(columns={"score": "score_per_id"})
)

# step 2: mean across ids for each agent
agent_score = (
    per_id.groupby("agent_name", as_index=False)["score_per_id"]
          .mean()
          .rename(columns={"score_per_id": "agent_score"})
)

agent_score.round(3)

,agent_name,agent_score
0,20250603_Refact_Agent_claude-4-sonnet,0.578
1,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,0.577
2,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,0.579
3,20250807_mini-v1.7.0_gpt-5-mini,0.430
4,20250928_trae_doubao_seed_code,0.549


In [57]:
round((agent_score_effect["effect_score"] + agent_score_intent["intent_score"])/2, 3)

0    0.578
1    0.577
2    0.579
3    0.430
4    0.549
dtype: float64

In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29700 entries, 0 to 7424
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   agent_name  29700 non-null  object 
 1   q_type      29700 non-null  object 
 2   trial       29700 non-null  int64  
 3   id_         29700 non-null  object 
 4   answer      29700 non-null  object 
 5   score       29700 non-null  float64
 6   resolved    29700 non-null  bool   
 7   truth       29700 non-null  object 
dtypes: bool(1), float64(1), int64(1), object(5)
memory usage: 1.8+ MB


In [58]:
# Basic checks
assert df.shape[1] == 8
assert set(["agent_name","q_type","trial","id_","answer","score","resolved","truth"]).issubset(df.columns)

# Cardinalities
n_agents = df["agent_name"].nunique()
n_ids = df["id_"].nunique()
n_q = df["q_type"].nunique()
n_trials = df["trial"].nunique()

print("agents:", n_agents, "ids:", n_ids, "q_types:", n_q, "trials:", n_trials)

# Expected total rows if complete grid:
expected = n_agents * n_ids * n_q * n_trials
print("expected rows:", expected, "actual rows:", len(df))

# Identify missing combinations (optional but recommended)
grid = (
    df[["agent_name","id_","q_type","trial"]]
    .drop_duplicates()
    .assign(present=True)
)
# If you want to explicitly find missing combos:
full = (
    pd.MultiIndex.from_product(
        [
            df["agent_name"].unique(),
            df["id_"].unique(),
            df["q_type"].unique(),
            df["trial"].unique(),
        ],
        names=["agent_name","id_","q_type","trial"]
    )
    .to_frame(index=False)
)
missing = full.merge(grid, how="left", on=["agent_name","id_","q_type","trial"])
missing = missing[missing["present"].isna()].drop(columns="present")
print("missing rows:", len(missing))


agents: 5 ids: 297 q_types: 4 trials: 5
expected rows: 29700 actual rows: 29700
missing rows: 0


In [59]:
per_item = (
    df.groupby(["agent_name", "id_", "q_type"], as_index=False)
      .agg(
          score_mean=("score", "mean"),
          resolved_any=("resolved", "max"),   # OR use mean if "resolved" varies by trial
          resolved_rate=("resolved", "mean"), # resolution frequency across trials
          n_trials=("trial", "nunique"),
      )
)

In [60]:
per_item

,agent_name,id_,q_type,score_mean,resolved_any,resolved_rate,n_trials
0,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-12907,ee_effect,1.0,True,1.0,5
1,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-12907,ee_intent,1.0,True,1.0,5
2,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-12907,local_effect,1.0,True,1.0,5
3,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-12907,local_intent,0.8,True,1.0,5
4,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-13033,ee_effect,0.4,False,0.0,5
...,...,...,...,...,...,...,...
5935,20250928_trae_doubao_seed_code,sympy__sympy-24539,local_intent,0.0,True,1.0,5
5936,20250928_trae_doubao_seed_code,sympy__sympy-24562,ee_effect,1.0,True,1.0,5
5937,20250928_trae_doubao_seed_code,sympy__sympy-24562,ee_intent,1.0,True,1.0,5
5938,20250928_trae_doubao_seed_code,sympy__sympy-24562,local_effect,0.8,True,1.0,5


In [61]:
print(sorted(df["q_type"].unique()))

# Fill this mapping with your exact labels
qtype_to_col = {
    "ee_intent": ("End-to-End", "Intent"),
    "ee_effect": ("End-to-End", "Effect"),
    "local_intent":      ("Local",      "Intent"),
    "local_effect":      ("Local",      "Effect"),
}


['ee_effect', 'ee_intent', 'local_effect', 'local_intent']


In [62]:
# Keep only q_types that are in the mapping (optional but safer)
mapped = per_item[per_item["q_type"].isin(qtype_to_col.keys())].copy()

# A simple agent x q_type mean table
agent_q = (
    mapped.groupby(["agent_name", "q_type"], as_index=False)
          .agg(score=("score_mean", "mean"))
)

# Pivot to columns
agent_pivot = agent_q.pivot(index="agent_name", columns="q_type", values="score")
agent_pivot = agent_pivot.rename(columns={k: f"{v[0]}|{v[1]}" for k, v in qtype_to_col.items()})
agent_pivot = agent_pivot.reindex(columns=[
    "End-to-End|Intent",
    "End-to-End|Effect",
    "Local|Intent",
    "Local|Effect",
])

In [63]:
agent_pivot

q_type,End-to-End|Intent,End-to-End|Effect,Local|Intent,Local|Effect
agent_name,,,,
20250603_Refact_Agent_claude-4-sonnet,0.722559,0.713805,0.354882,0.521886
20250720_Lingxi-v1.5_claude-4-sonnet-20250514,0.704377,0.715152,0.368350,0.519865
20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,0.722559,0.697643,0.352862,0.544781
20250807_mini-v1.7.0_gpt-5-mini,0.467340,0.507744,0.302357,0.441751
20250928_trae_doubao_seed_code,0.636364,0.716498,0.331987,0.511785


In [64]:
agent_pivot["Expl. Score"] = agent_pivot.mean(axis=1, skipna=True)

In [65]:
agent_pivot.round(3)

q_type,End-to-End|Intent,End-to-End|Effect,Local|Intent,Local|Effect,Expl. Score
agent_name,,,,,
20250603_Refact_Agent_claude-4-sonnet,0.723,0.714,0.355,0.522,0.578
20250720_Lingxi-v1.5_claude-4-sonnet-20250514,0.704,0.715,0.368,0.520,0.577
20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,0.723,0.698,0.353,0.545,0.579
20250807_mini-v1.7.0_gpt-5-mini,0.467,0.508,0.302,0.442,0.430
20250928_trae_doubao_seed_code,0.636,0.716,0.332,0.512,0.549


In [75]:
per_instance = (
    df.groupby(
        ["agent_name", "id_"],
        as_index=False
    )["score"]
    .mean()
    .rename(columns={"score": "explanation_score"})
)

In [76]:
per_instance

,agent_name,id_,explanation_score
0,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-12907,0.95
1,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-13033,0.35
2,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-13236,0.10
3,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-13453,0.60
4,20250603_Refact_Agent_claude-4-sonnet,astropy__astropy-13579,0.30
...,...,...,...
1480,20250928_trae_doubao_seed_code,sympy__sympy-24066,0.65
1481,20250928_trae_doubao_seed_code,sympy__sympy-24213,0.10
1482,20250928_trae_doubao_seed_code,sympy__sympy-24443,0.55
1483,20250928_trae_doubao_seed_code,sympy__sympy-24539,0.50


In [77]:
subset = per_instance[
    (per_instance["agent_name"].isin(["20250928_trae_doubao_seed_code", "20250805_openhands-Qwen3-Coder-480B-A35B-Instruct"]))
]

paired = (
    subset.pivot(
        index="id_",
        columns="agent_name",
        values="explanation_score"
    )
    .dropna()
)

In [78]:
paired

agent_name,20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,20250928_trae_doubao_seed_code
id_,,
astropy__astropy-12907,0.50,0.50
astropy__astropy-13033,0.60,0.90
astropy__astropy-13236,0.25,0.25
astropy__astropy-13453,0.55,0.55
astropy__astropy-13579,0.25,0.25
...,...,...
sympy__sympy-24066,0.50,0.65
sympy__sympy-24213,1.00,0.10
sympy__sympy-24443,0.45,0.55


In [79]:
from scipy.stats import wilcoxon

stat, p_value = wilcoxon(
    paired["20250928_trae_doubao_seed_code"],
    paired["20250805_openhands-Qwen3-Coder-480B-A35B-Instruct"],
    alternative="two-sided"
)

print(f"W={stat:.3f}, p={p_value:.4g}, N={len(paired)}")

W=11327.500, p=0.02521, N=297


In [80]:
per_instance = (
    df.groupby(
        ["agent_name", "trial"],
        as_index=False
    )["score"]
    .mean()
    .rename(columns={"score": "explanation_score"})
)

In [81]:
per_instance

,agent_name,trial,explanation_score
0,20250603_Refact_Agent_claude-4-sonnet,1,0.559764
1,20250603_Refact_Agent_claude-4-sonnet,2,0.580808
2,20250603_Refact_Agent_claude-4-sonnet,3,0.575758
3,20250603_Refact_Agent_claude-4-sonnet,4,0.579966
4,20250603_Refact_Agent_claude-4-sonnet,5,0.595118
5,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,1,0.558081
6,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,2,0.575758
7,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,3,0.573232
8,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,4,0.588384
9,20250720_Lingxi-v1.5_claude-4-sonnet-20250514,5,0.589226


In [82]:
per_instance.groupby("agent_name")["explanation_score"].std()

agent_name
20250603_Refact_Agent_claude-4-sonnet                0.012668
20250720_Lingxi-v1.5_claude-4-sonnet-20250514        0.012774
20250805_openhands-Qwen3-Coder-480B-A35B-Instruct    0.011350
20250807_mini-v1.7.0_gpt-5-mini                      0.013184
20250928_trae_doubao_seed_code                       0.019173
Name: explanation_score, dtype: float64

In [83]:
per_instance.groupby("agent_name")["explanation_score"].mean()

agent_name
20250603_Refact_Agent_claude-4-sonnet                0.578283
20250720_Lingxi-v1.5_claude-4-sonnet-20250514        0.576936
20250805_openhands-Qwen3-Coder-480B-A35B-Instruct    0.579461
20250807_mini-v1.7.0_gpt-5-mini                      0.429798
20250928_trae_doubao_seed_code                       0.549158
Name: explanation_score, dtype: float64

In [85]:
per_instance.groupby("agent_name")["explanation_score"].sem()

agent_name
20250603_Refact_Agent_claude-4-sonnet                0.005665
20250720_Lingxi-v1.5_claude-4-sonnet-20250514        0.005713
20250805_openhands-Qwen3-Coder-480B-A35B-Instruct    0.005076
20250807_mini-v1.7.0_gpt-5-mini                      0.005896
20250928_trae_doubao_seed_code                       0.008574
Name: explanation_score, dtype: float64

In [87]:
agent_pivot["sem"] = per_instance.groupby("agent_name")["explanation_score"].sem()

In [90]:
agent_pivot.round(4)

q_type,End-to-End|Intent,End-to-End|Effect,Local|Intent,Local|Effect,Expl. Score,sem
agent_name,,,,,,
20250603_Refact_Agent_claude-4-sonnet,0.7226,0.7138,0.3549,0.5219,0.5783,0.0057
20250720_Lingxi-v1.5_claude-4-sonnet-20250514,0.7044,0.7152,0.3684,0.5199,0.5769,0.0057
20250805_openhands-Qwen3-Coder-480B-A35B-Instruct,0.7226,0.6976,0.3529,0.5448,0.5795,0.0051
20250807_mini-v1.7.0_gpt-5-mini,0.4673,0.5077,0.3024,0.4418,0.4298,0.0059
20250928_trae_doubao_seed_code,0.6364,0.7165,0.3320,0.5118,0.5492,0.0086
